In [ ]:
# 01 · 통합 설정: 한 T4에서 두 공개 모델을 순서대로 비교
BACKENDS = ['smol', 'octo']
COMMON = {
    'smoke_updates': 3,
    'dev_episodes': 2,
    'test_episodes': 8,
    'max_steps': 200,
    'seed': 42,
    'device': 'cuda',
    'github_repository': 'SongYunu/moveBoxes',
    'project_ref': 'main',
    'project_dir': '/content/moveBoxes_foundation',
    'output_root': '/content/moveboxes_results',
    'env_root': '/content/moveboxes_envs_foundation',
    'data_dir': '/content/moveboxes_rgb_data',
    'download_cache': '/content/moveboxes_data_cache',
    'simulator_repo': '/content/berlin-marso-foundation',
    'simulator_commit': '6048f33217f26ae39009a812f53c81171517f393',
    'notebook_name': 'moveboxes_foundation_both_colab.ipynb',
}
MODEL = {
    'smol': dict(run_name='moveboxes_smolvla_rgb_full_v2',micro_batch=1,accumulate=6,
                 chunk=8,execute_steps=2,inference_steps=5,image_size=256,
                 lr=1e-4,updates=3000,eval_interval=1500),
    'octo': dict(run_name='moveboxes_octo_small_rgb_full_v2',micro_batch=3,accumulate=2,
                 chunk=4,execute_steps=2,inference_steps=20,image_size=256,
                 lr=3e-5,updates=5700,eval_interval=1900),
}
CFGS = {name: dict(COMMON, backend=name, **MODEL[name]) for name in BACKENDS}
for name,cfg in CFGS.items():
    assert cfg['micro_batch']>0 and cfg['accumulate']>0
    print(name,'· 실효 배치',cfg['micro_batch']*cfg['accumulate'],'·',cfg['updates'],'updates')
print('두 모델을 섞어 평균내지 않고 각각 공동 학습한 뒤 0.2/0.3/0.5 점수로 비교합니다.')


In [ ]:
# 02 · GitHub 코드 로드 (기존 폴더가 있으면 최신 project_ref로 갱신)
import importlib, os, subprocess, sys
from pathlib import Path

PROJECT = Path(COMMON['project_dir'])
if not (PROJECT/'.git').exists():
    subprocess.run(['git','clone','--filter=blob:none','https://github.com/SongYunu/moveBoxes.git',str(PROJECT)],check=True)
subprocess.run(['git','fetch','origin',COMMON['project_ref']],cwd=PROJECT,check=True)
subprocess.run(['git','checkout','--detach','FETCH_HEAD'],cwd=PROJECT,check=True)
COMMON['project_commit']=subprocess.check_output(['git','rev-parse','HEAD'],cwd=PROJECT,text=True).strip()
sys.path.insert(0,str(PROJECT))
importlib.invalidate_caches()
sys.modules.pop('foundation.code.foundation_experiment',None)
from foundation.code.foundation_experiment import FoundationExperiment
experiments={name:FoundationExperiment(dict(cfg,project_commit=COMMON['project_commit'])) for name,cfg in CFGS.items()}
print('사용 코드:',COMMON['project_commit'])


In [ ]:
# 03 · 두 결과 Release 복원/백업 연결
import getpass, os
if not os.environ.get('GH_TOKEN'):
    try:
        from google.colab import userdata
        os.environ['GH_TOKEN']=userdata.get('GH_TOKEN')
    except Exception:
        token=getpass.getpass('GitHub fine-grained token (Contents: read/write): ').strip()
        if not token: raise RuntimeError('결과 복원/저장용 GitHub token이 필요합니다.')
        os.environ['GH_TOKEN']=token

def run_each(method,*args):
    results,errors={},{}
    for name in BACKENDS:
        print('\n========',name.upper(),method,'========')
        try: results[name]=getattr(experiments[name],method)(*args)
        except Exception as exc:
            errors[name]=repr(exc);print(name,'실패:',repr(exc))
    if len(errors)==len(BACKENDS): raise RuntimeError(method+' 단계에서 두 백엔드가 모두 실패했습니다: '+repr(errors))
    return results,errors

run_each('connect')


In [ ]:
# 04 · SmolVLA(Python 3.12), Octo(Python 3.10), 시뮬레이터 환경 설치
install_results,install_errors=run_each('install')


In [ ]:
# 05 · RGB 데이터는 한 번 내려받고, 두 실험에서 같은 분할을 각각 검증
data_results,data_errors=run_each('prepare')


In [ ]:
# 06 · 각 공개 초기값으로 3 update + Easy 1회 실제 동작 확인
# 한 모델이 T4에서 실패해도 다른 모델 확인은 계속합니다.
check_results,check_errors=run_each('check')


In [ ]:
# 07 · 두 모델 순차 공동 학습; 각 eval_interval마다 GitHub에 진행 adapter 저장
train_results,train_errors=run_each('train')


In [ ]:
# 08 · 각 모델의 최고 adapter로 Easy 테스트
easy_results,easy_errors=run_each('test','easy')


In [ ]:
# 09 · 각 모델의 최고 adapter로 Medium 테스트
medium_results,medium_errors=run_each('test','medium')


In [ ]:
# 10 · 각 모델의 최고 adapter로 Hard 테스트
hard_results,hard_errors=run_each('test','hard')


In [ ]:
# 11 · 모델별 세 난이도 가중 개발 점수 비교
import json
comparison={}
for name,experiment in experiments.items():
    state=experiment.report();best=state.get('best')
    comparison[name]=None if not best else dict(weighted_score=best['score'],scores=best['scores'],folder=best['folder'])
valid={k:v for k,v in comparison.items() if v is not None}
print(json.dumps(comparison,indent=2,ensure_ascii=False))
if valid:
    winner=max(valid,key=lambda k:valid[k]['weighted_score'])
    print('현재 고정 개발 seed 기준 후보:',winner,valid[winner])
else:
    print('아직 비교할 완성 checkpoint가 없습니다.')
